In [20]:
# Librerías para manipulación, visualización y carga portable de datos.
import pandas as pd
# import numpy as np 
# import matplotlib.pyplot as plt
# import seaborn as sns
import os

# Usamos una ruta para Kaggle y otra para ejecución local en VS Code.
kaggle_path = '/kaggle/input/california-housing-prices/housing.csv'
local_path = 'housing.csv'

# Cargamos el dataset desde la ruta que exista en el entorno actual.
if os.path.exists(kaggle_path):
    df = pd.read_csv(kaggle_path)
    print("Cargado desde el entorno de Kaggle")
elif os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print("Cargado desde el entorno local (VS Code)")
else:
    raise FileNotFoundError("No se encontro el archivo 'housing.csv' en ninguna de las rutas.")

# Vista inicial: permite comprobar que las columnas se cargaron correctamente.
df.head()

In [21]:
df.info()

In [22]:
print("\nValores nulos por columna:")
print(df.isnull().sum())

In [23]:
df.describe()

In [24]:
from sklearn.model_selection import train_test_split

# Separamos features (X) del objetivo (y)
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

# Split SIN trampas: 80% train, 20% test, con semilla fija para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]} filas")
print(f"Test: {X_test.shape[0]} filas")


In [25]:
# Calcular la mediana SOLO con train
mediana_bedrooms = X_train["total_bedrooms"].median()
print(f"Mediana de total_bedrooms (train): {mediana_bedrooms}")

# Rellenamos los nulos en ambos conjuntos con esa mediana
X_train["total_bedrooms"] = X_train["total_bedrooms"].fillna(mediana_bedrooms)
X_test["total_bedrooms"] = X_test["total_bedrooms"].fillna(mediana_bedrooms)

# Verificamos que no quedan nulos
print("Nulos en train: ", X_train["total_bedrooms"].isnull().sum())
print("Nulos en train: ", X_train["total_bedrooms"].isnull().sum())

In [26]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# fit SOLO con train
encoder.fit(X_train[["ocean_proximity"]])

def aplicar_encoding(X, encoder):
    codificado = pd.DataFrame(
        encoder.transform(X[["ocean_proximity"]]),
        columns=encoder.get_feature_names_out(["ocean_proximity"]),
        index=X.index,
    )
    return pd.concat([X.drop(columns=["ocean_proximity"]), codificado], axis=1)

X_train = aplicar_encoding(X_train, encoder)
X_test = aplicar_encoding(X_test, encoder)

print(X_train.columns.tolist())
print(X_train.shape, X_test.shape)

In [27]:
def agregar_features(X):
    X = X.copy()
    X["rooms_per_household"] = X["total_rooms"] / X["households"]
    X["bedrooms_per_room"] = X["total_bedrooms"] / X["total_rooms"]
    X["population_per_household"] = X["population"] / X["households"]
    return X

X_train = agregar_features(X_train)
X_test = agregar_features(X_test)

print(X_train.shape, X_test.shape)
print(X_train[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe())